# MyStepLR: simple learning rate scheduler

Implements a minimal step LR scheduler that works with the `MySGD` optimizer from Lecture 3.
The scheduler multiplies the learning rate by `gamma` every `step_size` epochs.

In [1]:
import torch
from torch import nn, Tensor
from typing import Iterable, Optional

# separate seed from optimizer notebook to show independent runs
torch.manual_seed(1)


In [2]:
class MySGD:
    """Same MySGD as in the optimizer notebook, kept here for a self-contained demo.
    Includes optional weight decay and gradient clipping.
    """
    def __init__(self, params: Iterable[Tensor], lr: float = 0.1, weight_decay: float = 0.0, grad_clip: Optional[float] = None):
        self.params = [p for p in params if p.requires_grad]
        if not self.params:
            raise ValueError('No trainable parameters provided to MySGD.')
        self.lr = lr
        self.weight_decay = weight_decay
        self.grad_clip = grad_clip

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def _clip_grads(self):
        if self.grad_clip is None:
            return
        grads = [p.grad for p in self.params if p.grad is not None]
        if not grads:
            return
        total_norm = torch.norm(torch.stack([g.norm() for g in grads]))
        if total_norm > self.grad_clip:
            scale = self.grad_clip / (total_norm + 1e-8)
            for g in grads:
                g.mul_(scale)

    @torch.no_grad()
    def step(self):
        self._clip_grads()
        for p in self.params:
            if p.grad is None:
                continue
            grad = p.grad
            if self.weight_decay != 0.0:
                grad = grad + self.weight_decay * p.data
            p.data.add_(grad, alpha=-self.lr)


In [3]:
class MyStepLR:
    """Step LR scheduler that decays optimizer.lr every `step_size` epochs by `gamma`.

    Mimics torch.optim.lr_scheduler.StepLR but operates directly on the optimizer's lr field.
    """
    def __init__(self, optimizer: MySGD, step_size: int, gamma: float = 0.1):
        self.optimizer = optimizer
        self.step_size = step_size
        self.gamma = gamma
        self.last_epoch = 0
        self.history = [optimizer.lr]  # track LR for inspection/plotting

    def step(self):
        # Called once per epoch; decay happens on multiples of step_size
        self.last_epoch += 1
        if self.last_epoch % self.step_size == 0:
            self.optimizer.lr *= self.gamma
        self.history.append(self.optimizer.lr)


## Demo: binary classification with LR decay
Train a logistic regression classifier on a toy 2D dataset, decaying LR every 10 epochs.

In [4]:
# toy dataset: two Gaussian clusters in 2D
N = 200
mean0, mean1 = torch.tensor([-1.0, -1.0]), torch.tensor([1.0, 1.0])
X0 = mean0 + 0.6 * torch.randn(N // 2, 2)
X1 = mean1 + 0.6 * torch.randn(N // 2, 2)
X = torch.cat([X0, X1], dim=0)
y = torch.cat([torch.zeros(N // 2), torch.ones(N // 2)]).unsqueeze(1)

model = nn.Sequential(nn.Linear(2, 1))
criterion = nn.BCEWithLogitsLoss()
optimizer = MySGD(model.parameters(), lr=0.4, weight_decay=0.0, grad_clip=5.0)
scheduler = MyStepLR(optimizer, step_size=10, gamma=0.3)

loss_hist = []
for epoch in range(30):
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()      # optimize with current LR
    scheduler.step()      # possibly decay LR for next epoch
    loss_hist.append(loss.item())
    if (epoch + 1) % 5 == 0:
        with torch.no_grad():
            preds = (torch.sigmoid(model(X)) > 0.5).float()
            acc = (preds == y).float().mean().item()
        print(f'Epoch {epoch+1:02d} | loss={loss.item():.4f} | lr={optimizer.lr:.4f} | acc={acc:.3f}')

print('LR schedule:', scheduler.history)


Epoch 05 | loss=0.2852 | lr=0.4000 | acc=0.990
Epoch 10 | loss=0.1885 | lr=0.1200 | acc=0.990
Epoch 15 | loss=0.1676 | lr=0.1200 | acc=0.990
Epoch 20 | loss=0.1566 | lr=0.0360 | acc=0.990
Epoch 25 | loss=0.1524 | lr=0.0360 | acc=0.990
Epoch 30 | loss=0.1497 | lr=0.0108 | acc=0.990
LR schedule: [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.036, 0.036, 0.036, 0.036, 0.036, 0.036, 0.036, 0.036, 0.036, 0.036, 0.010799999999999999]


## Notes
- Learning rate drops at epochs 10, 20, 30 (step_size=10, gamma=0.3).
- You can swap `MyStepLR` for a warmup-style scheduler by ramping `optimizer.lr` up instead of down in `step()`.
- The optimizer/scheduler APIs mirror the Lecture 3 design: `optimizer.step()` then `scheduler.step()` each epoch.